In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adbstoragevd001";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL adbcredentialvd001)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL adbcredentialvd001)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL adbcredentialvd001)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL adbcredentialvd001)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL adbcredentialvd001)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.bronze_aisle (
aisle_id integer,
aisle string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/bronze_aisle"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.bronze_department (
department_id integer,
department string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/bronze_department"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.bronze_order_products__prior (
  order_id integer,
  product_id integer,
  add_to_cart_order integer,
  reordered integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/bronze_order_products__prior"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.bronze_order_products__train (
  order_id integer,
  product_id integer,
  add_to_cart_order integer,
  reordered integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/bronze_order_products__train"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.bronze_orders (
  order_id integer,
  user_id integer,
  eval_set string,
  order_number integer,
  order_dow integer,
  order_hour_of_day integer,
  days_since_prior_order float,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/bronze_orders"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.bronze_products (
  product_id integer,
  product_name string,
  aisle_id integer,
  department_id integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/bronze_products"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.bronze_sample_submission (
  order_id integer,
  products integer,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/bronze_sample_submission"

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.silver_orders (
    order_id integer,
    user_id integer,
    order_number integer,
    order_dow integer,
    order_hour_of_day integer,
    days_since_prior_order float,
    order_day_name string,
    order_time_category string
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/silver_orders"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.silver_order_products (
    order_id integer,
    product_id integer,
    add_to_cart_order integer,
    reordered integer,
    cart_position_category string,
    reorder_flag string,
    order_type string
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/silver_order_products"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.silver_Products (
    product_id integer,
    product_name string,
    aisle_id integer,
    aisle string,
    department_id integer,
    department string,
    product_category_group string
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/silver_Products"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.silver_orders_products_detail (
    order_id integer,
    user_id integer,
    order_number integer,
    order_dow integer,
    order_hour_of_day integer,
    days_since_prior_order float,
    order_day_name string,
    order_time_category string,
    product_id integer,
    product_name string,
    aisle_id integer,
    aisle string,
    department_id integer,
    department string,
    product_category_group string,
    add_to_cart_order integer,
    reordered  integer,
    cart_position_category string,
    reorder_flag string
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/silver_orders_products_detail"

###Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.golden_top_products (
  product_id integer,
  product_name string,
  department string,
  aisle string,
  total_orders integer,
  total_reorders integer
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_top_products"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.gold_reorder_rate_department  (
  department string,
  total_items integer,
  total_reorders integer,
  reorder_rate_pct float
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/gold_reorder_rate_department"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.gold_purchase_patterns (
  order_day_name string,
  order_time_category string,
  total_orders integer,
  total_items integer
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/gold_purchase_patterns"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.gold_user_segments (
  total_orders integer,
  avg_days_between_orders float,
  user_segment string
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/gold_user_segments"